In [3]:
import os
from scipy.ndimage import gaussian_filter
import numpy as np
from os.path import getsize
import napari
from skimage.io import imread

filename =  'Image_001_001.raw'
previewFilename = 'ChanC_Preview.tif'

class thorlabsFile():
   
    def __init__(self,folder) -> None:

        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)

        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] #change with self determination
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width))
        self.app = napari.Viewer()
        #self.app.add_image(self.array)

    def loadFile(self,folder):

        try:
            self.r.close()
        except:
            pass
        
        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)
        prev = imread(os.path.join('./',previewFilename))

        self.width = prev.shape[1] #change with self determination
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width))

        l = self.app.layers['Image']
        l.data = self.array
        
    def getImage(self,n):

        offset = n*self.frameSize
        self.r.seek(offset)
        st = self.r.read(self.frameSize)
        nparray = np.frombuffer(st,dtype = np.uint16).reshape((1,self.height,self.width))
        
        return nparray

    def loadWholeStack(self,start=0,end=-1,step=1):

        
        if end == -1:
            end = nFrames
        totalSize = end-start

        #stack = np.zeros((self.height,self.width,totalSize),dtype=np.uint16)
        #for i in range(start,end,step):
        #    stack[:,:,i-start] = gaussian_filter(self.getImage(i),2)
        #    #stack.append(getImage(r,i,width,height))
        
        offset = start*self.frameSize
        self.r.seek(offset)
        st = self.r.read(self.frameSize*totalSize)
        stack = np.frombuffer(st,dtype = np.uint16).reshape((totalSize,self.height,self.width))
        stack2 = stack.copy()
        stack2.setflags(write=1)

      #  stack = stack.swapaxes(0,2)
      #  stack = stack.swapaxes(1,2)
        

        #for i in range(stack2.shape[1]):
        #    stack2[:,i,:] = gaussian_filter(stack2[:,i,:],(2,0))
        stack2 = gaussian_filter(stack2,(2,2,2))
        return stack2

    def loadNextNFrames(self,n):

        self.array= np.vstack([self.array, self.loadWholeStack(start=self.currentLastFrame,end = self.currentLastFrame+n)])

        try:
            l = self.app.layers['Image']
            l.data = self.array
        except:
            self.app.add_image(self.array)

        self.currentLastFrame = self.array.shape[0]

    def loadUpToFrameN(self,n):
        
        if (n<self.currentLastFrame) | (n>self.nFrames):
            return

        self.array= np.vstack([self.array, self.loadWholeStack(start=self.currentLastFrame,end = n)])

        if self.currentLastFrame == 0:
                self.app.add_image(self.array)
        else:
            l = self.app.layers['Image']
            l.data = self.array
        self.currentLastFrame = self.array.shape[0]



In [4]:
tb = thorlabsFile('../sampleImage/')

# Use this to load N extra frames

In [5]:
tb.loadNextNFrames(1000)

# Use this to load up to frame N

In [6]:
tb.loadUpToFrameN(1000)

# Use this to load another file

tb.loadFile('./') 